# RocketPy Full Pipeline — Flight Thrust Reconstruction & Simulation

This notebook reconstructs the flight thrust curve from onboard pressure data and runs a RocketPy trajectory simulation.

**Pipeline:**
1. Load flight data (`Flight_Complete_Burn.csv` — already retimed) and SMT0033 ground test
2. Calibrate Pc → Thrust using SMT0033 (load cell reference)
3. Reconstruct flight thrust from onboard Pc
4. Validate metrics (total impulse, motor class)
5. Export `.eng` file and run RocketPy simulation

**Data sources:**
| Dataset | Rate | Sensors | Purpose |
|---------|------|---------|---------| 
| Flight onboard | 2.45 Hz | Pc, Ptank | Thrust reconstruction (this notebook) |
| SMT0033 | 50 Hz | Pc, Ptank, **thrust (load cell)** | Calibration reference |

**No ignition synthesis** — the first flight sample is at t ≈ 0.4 s; we do NOT insert a synthetic ignition transient from hot-fire tests.

In [ ]:
!pip install -q rocketpy numpy matplotlib pandas

In [ ]:
import os, subprocess

# When opened from GitHub in Colab, the repo is not cloned automatically
if not os.path.exists('data/Flight_Complete_Burn.csv'):
    if not os.path.exists('hybrid-rocket-trajectory'):
        subprocess.check_call(
            'git clone https://github.com/jmartos-br/hybrid-rocket-trajectory.git',
            shell=True,
        )
    os.chdir('hybrid-rocket-trajectory')
    subprocess.check_call('git checkout rocketpy-retimed-motor', shell=True)

print('CWD:', os.getcwd())
print('Data files:', sorted(os.listdir('data')))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

## 1 · Load Data

**Flight data** (`Flight_Complete_Burn.csv`): Onboard Pc and Ptank at 2.45 Hz, time axis already corrected via Ptank blowdown correlation. Ignition (Pc > 2 bar) at t ≈ 0.408 s.

**SMT0033** (`SMT0033_extracted.csv`): Static motor test with load cell. Thrust recorded in daN (×10 → Newtons). Ignition aligned at Pc > 2 bar, pre-ignition thrust baseline subtracted.

In [ ]:
# ── Flight data (already retimed) ──
flight_raw = pd.read_csv('data/Flight_Complete_Burn.csv', sep=';')
print(f"Flight: {len(flight_raw)} rows")
print(f"  Time range: {flight_raw['time_real_s'].min():.1f} to {flight_raw['time_real_s'].max():.1f} s")
print(f"  Pc range:   {flight_raw['pc_bar'].min():.2f} – {flight_raw['pc_bar'].max():.2f} bar")
print(f"  Ptank range: {flight_raw['ptank_bar'].min():.2f} – {flight_raw['ptank_bar'].max():.2f} bar")

# ── SMT0033 (ground test with load cell) ──
smt_raw = pd.read_csv('data/SMT0033_extracted.csv')
smt = smt_raw.copy()
smt['time_s'] = smt['time_ms'] / 1000.0

# Align ignition: t = 0 when Pc first exceeds 2 bar
t_ign = smt.loc[smt['chamber_pressure_bar'] > 2.0, 'time_s'].iloc[0]
smt['time_s'] -= t_ign

# Convert thrust: subtract pre-ignition baseline, daN → N
pre = smt['time_s'] < 0
thrust_baseline = smt.loc[pre, 'thrust_daN'].median()
smt['thrust_N'] = np.maximum((smt['thrust_daN'] - thrust_baseline) * 10.0, 0.0)

print(f"\nSMT0033: {len(smt)} rows")
print(f"  Time range: {smt['time_s'].min():.1f} to {smt['time_s'].max():.1f} s")
print(f"  Max thrust: {smt['thrust_N'].max():.1f} N")
print(f"  Ptank_0:    {smt.loc[pre, 'tank_pressure_bar'].median():.1f} bar")

## 2 · Flight Overview

Onboard Pc and Ptank vs time. Note the jump from t = 0 (Pc ≈ 0.6 bar) to t = 0.408 s (Pc ≈ 33.7 bar) — the ignition transient happened between samples and was missed by the 2.45 Hz logger.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

t = flight_raw['time_real_s']

ax1.plot(t, flight_raw['pc_bar'], 'b.-', ms=4, lw=1)
ax1.axhline(2.0, color='r', ls='--', alpha=0.5, label='Pc = 2 bar (ignition threshold)')
ax1.set_ylabel('Chamber Pressure [bar]')
ax1.set_title('Flight Onboard Data (retimed)')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(t, flight_raw['ptank_bar'], 'g.-', ms=4, lw=1)
ax2.set_ylabel('Tank Pressure [bar]')
ax2.set_xlabel('Time [s]')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3 · SMT0033 Ground Test

Static motor test with load cell. This is our calibration reference — the **only** dataset with both Pc and thrust measured simultaneously under the same motor design.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
mask = (smt['time_s'] > -1) & (smt['time_s'] < 28)
s = smt[mask]

axes[0].plot(s['time_s'], s['thrust_N'], 'r-', lw=0.8)
axes[0].set_ylabel('Thrust [N]')
axes[0].set_title('SMT0033 — Static Motor Test (50 Hz, Load Cell)')
axes[0].grid(alpha=0.3)

axes[1].plot(s['time_s'], s['chamber_pressure_bar'], 'b-', lw=0.8)
axes[1].set_ylabel('Pc [bar]')
axes[1].grid(alpha=0.3)

axes[2].plot(s['time_s'], s['tank_pressure_bar'], 'g-', lw=0.8)
axes[2].set_ylabel('Ptank [bar]')
axes[2].set_xlabel('Time [s]')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4 · Pc → Thrust Calibration

Calibrate chamber pressure to thrust using SMT0033.

$$Pc_{gauge} = \max(Pc - Pc_0,\; 0)$$

Two fits:
- **Origin:** $$F = a \cdot Pc_{gauge}$$ — physically motivated (zero thrust at zero pressure)
- **Intercept:** $$F = a \cdot Pc_{gauge} + b$$ — better R², captures real nozzle/injector offset

We use **steady-burn data only** (t > 0.5 s, Pc_gauge > 1 bar, thrust > 5 N) to avoid the noisy ignition transient.

In [ ]:
# Pc baseline from pre-ignition
pc0_smt = smt.loc[smt['time_s'] < 0, 'chamber_pressure_bar'].median()
smt['pc_gauge'] = np.maximum(smt['chamber_pressure_bar'] - pc0_smt, 0.0)

# Steady-burn mask
cal_mask = (smt['time_s'] > 0.5) & (smt['pc_gauge'] > 1.0) & (smt['thrust_N'] > 5.0)
pcg = smt.loc[cal_mask, 'pc_gauge'].values
thr = smt.loc[cal_mask, 'thrust_N'].values

# ── Origin fit: F = a · Pc_gauge ──
a_origin = float(np.dot(pcg, thr) / np.dot(pcg, pcg))
yhat_o = a_origin * pcg
ss_res_o = np.sum((thr - yhat_o)**2)
ss_tot = np.sum((thr - np.mean(thr))**2)
r2_origin = 1 - ss_res_o / ss_tot

# ── Intercept fit: F = a · Pc_gauge + b ──
A = np.vstack([pcg, np.ones_like(pcg)]).T
(a_inter, b_inter), *_ = np.linalg.lstsq(A, thr, rcond=None)
a_inter, b_inter = float(a_inter), float(b_inter)
yhat_i = a_inter * pcg + b_inter
ss_res_i = np.sum((thr - yhat_i)**2)
r2_inter = 1 - ss_res_i / ss_tot

print(f"Origin fit:    F = {a_origin:.3f} · Pc_gauge            (R² = {r2_origin:.5f})")
print(f"Intercept fit: F = {a_inter:.3f} · Pc_gauge + ({b_inter:.3f})  (R² = {r2_inter:.5f})")
print(f"\n→ Using intercept fit (better R²).")

# ── Plot ──
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(pcg, thr, s=4, alpha=0.3, c='gray', label='SMT0033 data')
x = np.linspace(0, pcg.max() * 1.05, 200)
ax.plot(x, a_origin * x, 'r-', lw=2,
        label=f'Origin: F = {a_origin:.2f}·Pc  (R²={r2_origin:.4f})')
ax.plot(x, a_inter * x + b_inter, 'b--', lw=2,
        label=f'Intercept: F = {a_inter:.2f}·Pc + {b_inter:.1f}  (R²={r2_inter:.4f})')
ax.set_xlabel('Pc_gauge [bar]')
ax.set_ylabel('Thrust [N]')
ax.set_title('Pc → Thrust Calibration (SMT0033)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5 · Reconstruct Flight Thrust

Apply the intercept calibration to flight Pc. The first thrust point is at t ≈ 0.408 s — the true ignition peak (which happened in < 400 ms) is **not captured** by the 2.45 Hz logger.

In [ ]:
# Flight Pc baseline (pre-ignition)
pc0_flight = flight_raw.loc[flight_raw['time_real_s'] < 0, 'pc_bar'].median()
print(f"Pc baseline — flight: {pc0_flight:.3f} bar, SMT0033: {pc0_smt:.3f} bar")

# Compute Pc_gauge and reconstruct thrust
flight = flight_raw.copy()
flight['pc_gauge'] = np.maximum(flight['pc_bar'] - pc0_flight, 0.0)
flight['thrust_N'] = a_inter * flight['pc_gauge'] + b_inter

# Zero out pre-ignition and clamp negatives
flight.loc[flight['pc_bar'] < 2.0, 'thrust_N'] = 0.0
flight['thrust_N'] = np.maximum(flight['thrust_N'], 0.0)

# Burn region
burn = flight[flight['thrust_N'] > 0].copy()

# ── Plot ──
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(burn['time_real_s'], burn['thrust_N'], 'ro-', ms=5, lw=1.2,
        label='Flight thrust (reconstructed)')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Thrust [N]')
ax.set_title('Flight Thrust Curve — No Ignition Synthesis')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nBurn samples: {len(burn)}")
print(f"First: t = {burn['time_real_s'].iloc[0]:.3f} s, F = {burn['thrust_N'].iloc[0]:.1f} N")
print(f"Last:  t = {burn['time_real_s'].iloc[-1]:.3f} s, F = {burn['thrust_N'].iloc[-1]:.1f} N")

## 6 · Validation — Flight vs SMT0033

| Plot | What it shows |
|------|---------------|
| **Left** — Thrust vs Time | Both should show a regressive profile (high initial, decaying with Ptank) |
| **Right** — Thrust vs Ptank | Removes timing — shows the fundamental blowdown operating curve. Should overlap if same motor design. |

In [ ]:
# ── Metrics ──
t_burn = burn['time_real_s'].values
f_burn = burn['thrust_N'].values
total_impulse = float(np.trapz(f_burn, t_burn))
burn_duration = t_burn[-1] - t_burn[0]
avg_thrust = total_impulse / burn_duration
max_thrust = f_burn.max()
letter_idx = int(math.log2(total_impulse / 2.5))
motor_letter = chr(ord('A') + letter_idx)

smt_burn = smt[smt['thrust_N'] > 5]
smt_impulse = float(np.trapz(smt_burn['thrust_N'], smt_burn['time_s']))
smt_max = smt_burn['thrust_N'].max()

print("=" * 55)
print("           FLIGHT MOTOR METRICS")
print("=" * 55)
print(f"  Total impulse:   {total_impulse:>8.1f} N·s")
print(f"  Max thrust:      {max_thrust:>8.1f} N")
print(f"  Avg thrust:      {avg_thrust:>8.1f} N")
print(f"  Burn duration:   {burn_duration:>8.2f} s")
print(f"  Motor class:     {motor_letter}{int(avg_thrust)}")
print("-" * 55)
print(f"  SMT0033 impulse: {smt_impulse:>8.1f} N·s (reference)")
print(f"  SMT0033 max:     {smt_max:>8.1f} N")
print("=" * 55)

# ── Comparison plots ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Thrust vs Time
smt_p = smt[(smt['time_s'] > -0.5) & (smt['time_s'] < 28)]
ax1.plot(smt_p['time_s'], smt_p['thrust_N'], 'b-', alpha=0.7, lw=0.8, label='SMT0033 (measured)')
ax1.plot(burn['time_real_s'], burn['thrust_N'], 'ro-', ms=4, label='Flight (reconstructed)')
ax1.set_xlabel('Time [s]')
ax1.set_ylabel('Thrust [N]')
ax1.set_title('Thrust vs Time')
ax1.legend()
ax1.grid(alpha=0.3)

# Thrust vs Ptank
smt_bm = (smt['time_s'] > 0) & (smt['thrust_N'] > 5)
smt_s = smt[smt_bm].sort_values('tank_pressure_bar', ascending=False)
burn_s = burn.sort_values('ptank_bar', ascending=False)

ax2.plot(smt_s['tank_pressure_bar'], smt_s['thrust_N'], 'b-', alpha=0.7, lw=0.8, label='SMT0033')
ax2.plot(burn_s['ptank_bar'], burn_s['thrust_N'], 'ro-', ms=4, label='Flight')
ax2.invert_xaxis()
ax2.set_xlabel('Tank Pressure [bar]  →  blowdown')
ax2.set_ylabel('Thrust [N]')
ax2.set_title('Thrust vs Ptank (blowdown correlation)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7 · Export `.eng` File

Export the reconstructed thrust curve in RASP `.eng` format for RocketPy, OpenRocket, and RASAero.

In [ ]:
# Prepare: sort by time, shift so t=0 at first burn sample
eng = burn.sort_values('time_real_s')
t_eng = eng['time_real_s'].values.copy()
f_eng = eng['thrust_N'].values.copy()
t_eng = t_eng - t_eng[0]  # t=0 at motor ignition

# Append zero-thrust endpoint
t_eng = np.append(t_eng, t_eng[-1] + 0.01)
f_eng = np.append(f_eng, 0.0)

# Final metrics
impulse_eng = float(np.trapz(f_eng, t_eng))
avg_eng = impulse_eng / t_eng[-2]
letter_eng = chr(ord('A') + int(math.log2(impulse_eng / 2.5)))
designation = f'{letter_eng}{int(avg_eng)}'

eng_path = 'data/Flight_Reconstructed_NoIgn.eng'
with open(eng_path, 'w') as f:
    f.write(f'; Flight Reconstructed Thrust Curve (No Ignition Synthesis)\n')
    f.write(f'; Calibration: F = {a_inter:.3f} * Pc_gauge + ({b_inter:.3f})\n')
    f.write(f'; Total Impulse: {impulse_eng:.1f} Ns, Max Thrust: {f_eng.max():.1f} N, '
            f'Avg Thrust: {avg_eng:.1f} N, Burn Time: {t_eng[-2]:.1f} s\n')
    f.write(f'{designation} 100 1330 0 2.300 9.200 FlightRecon\n')
    for ti, fi in zip(t_eng, f_eng):
        f.write(f'  {ti:.4f}    {fi:.3f}\n')

print(f'Wrote: {eng_path}')
print(f'  Designation: {designation}')
print(f'  Burn time:   {t_eng[-2]:.2f} s')
print(f'  Impulse:     {impulse_eng:.1f} Ns')
print(f'  Max thrust:  {f_eng.max():.1f} N')
print(f'  Avg thrust:  {avg_eng:.1f} N')

## 8 · RocketPy Trajectory Simulation

| Parameter | Value |
|-----------|-------|
| Diameter | 100 mm |
| Total length | ~2.6 m |
| Airframe dry mass | 3.78 kg |
| Motor dry mass | 6.9 kg |
| Propellant mass | 2.42 kg |
| Launch rail | 7 m, 83.5° inclination |
| Parachute Cd×S | 5.78 m² (main, at apogee) |

In [ ]:
from rocketpy import Environment, Rocket, Flight, GenericMotor

# ── Environment: launch site near Abu Dhabi, 13 Feb 2026 ──
env = Environment(latitude=24.18133, longitude=53.688379, elevation=5)
env.set_date((2026, 2, 13, 12))
env.set_atmospheric_model(
    type='custom_atmosphere',
    wind_u=[(0, 0.07), (10, 0.07), (135, 0.00), (818, -0.45),
            (1542, -0.62), (3164, -0.51), (5854, 12.69)],
    wind_v=[(0, -4.00), (10, -4.00), (135, -4.19), (818, -1.46),
            (1542, 1.40), (3164, -0.51), (5854, 4.62)],
    pressure=[(0, 101500), (135, 100000), (818, 92500),
              (1542, 85000), (3164, 70000), (5854, 50000)],
    temperature=[(0, 302.95), (135, 301.65), (818, 295.25),
                 (1542, 288.75), (3164, 281.35), (5854, 264.25)],
)

# ── Motor ──
motor = GenericMotor(
    thrust_source=eng_path,
    burn_time=float(t_eng[-2]),
    chamber_radius=0.05,
    chamber_height=1.33,
    chamber_position=1.33 / 2,
    propellant_initial_mass=2.42,
    nozzle_radius=0.025,
    dry_mass=6.9,
    dry_inertia=(0.5, 0.5, 0.01),
    nozzle_position=0.0,
    center_of_dry_mass_position=1.33 / 2,
    coordinate_system_orientation='nozzle_to_combustion_chamber',
)

# ── Rocket ──
rocket = Rocket(
    radius=0.05,
    mass=3.780,
    inertia=(3.5, 3.5, 0.005),
    power_off_drag='data/poweroff_drag.csv',
    power_on_drag='data/poweron_drag.csv',
    center_of_mass_without_motor=1.869,
    coordinate_system_orientation='tail_to_nose',
)
rocket.add_motor(motor, position=0.0)
rocket.add_nose(length=0.3, kind='ogive', position=2.600)
rocket.add_trapezoidal_fins(
    n=4, root_chord=0.145, tip_chord=0.065, span=0.08,
    sweep_length=0.11, cant_angle=0.5, position=0.145,
)
rocket.add_tail(
    top_radius=0.05, bottom_radius=0.03, length=0.055, position=0.0,
)
rocket.set_rail_buttons(
    upper_button_position=1.80, lower_button_position=0.40, angular_position=88,
)

# ── Parachute ──
def main_trigger(p, h, y):
    return True if y[5] < 0 else False

rocket.add_parachute(
    name='Main',
    cd_s=2.2 * np.pi * (1.8288 / 2) ** 2,
    trigger=main_trigger,
    sampling_rate=105,
    lag=1.5,
    noise=(0, 8.3, 0.5),
)

# ── Simulate ──
sim = Flight(
    rocket=rocket,
    environment=env,
    rail_length=7.0,
    inclination=83.5,
    heading=90,
    max_time=600,
    time_overshoot=True,
)

print(f'Apogee AGL:       {float(sim.apogee - env.elevation):.0f} m')
print(f'Max speed:         {float(sim.max_speed):.1f} m/s')
print(f'Max acceleration:  {float(sim.max_acceleration):.1f} m/s²')
print(f'Time of apogee:    {float(sim.apogee_time):.1f} s')
print(f'Impact time:       {float(sim.t_final):.1f} s')

## 9 · Full Simulation Results

In [ ]:
sim.all_info()

## 10 · Download Results (Colab)

Zip and download all generated `.eng` files and plots.

In [ ]:
import glob, zipfile, shutil

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

out_dir = 'outputs_colab'
os.makedirs(out_dir, exist_ok=True)

for p in glob.glob('data/*.eng') + glob.glob('*.png'):
    try:
        shutil.copy2(p, os.path.join(out_dir, os.path.basename(p)))
    except Exception:
        pass

zip_path = 'outputs_colab.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for root, _, fnames in os.walk(out_dir):
        for fn in fnames:
            full = os.path.join(root, fn)
            z.write(full, arcname=os.path.relpath(full, '.'))

print(f'Wrote {zip_path}')
if IN_COLAB:
    colab_files.download(zip_path)